# Sanatan Beats — free गाना test (ACE-Step 1.5)

**कैसे चलाएँ:** ऊपर menu में **Runtime → Change runtime type → T4 GPU → Save**, फिर **Runtime → Run all**। करीब 8–12 मिनट। आख़िर में 4 गाने (2 छठ, 2 हनुमान) यहीं सुनाई देंगे।

In [ ]:
#@title 1 · GPU जाँच
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout[:600] if r.returncode == 0 else '❌ GPU नहीं मिला — Runtime → Change runtime type → T4 GPU चुनें, फिर Run all')

In [ ]:
#@title 2 · ACE-Step install (4–6 मिनट)
!git clone -q --depth 1 https://github.com/ACE-Step/ACE-Step-1.5.git /content/ace
%cd /content/ace
!pip install -q -e . 2>&1 | tail -3
%cd /content
print('✅ install done')

In [ ]:
#@title 3 · गाने की script
script = r"""
import os, sys, json, time, traceback
sys.path.insert(0, '/content/ace')
from acestep.handler import AceStepHandler
from acestep.llm_inference import LLMHandler
from acestep.inference import GenerationParams, GenerationConfig, generate_music
jobs = json.load(open('/content/jobs.json', encoding='utf-8'))
dit = AceStepHandler()
print(dit.initialize_service(project_root='/content/ace', config_path='acestep-v15-turbo', device='cuda', offload_to_cpu=True))
llm = LLMHandler(); use_llm = False
try:
    res = llm.initialize(checkpoint_dir='/content/ace/checkpoints', lm_model_path='acestep-5Hz-lm-1.7B', backend='pt', device='cuda')
    print('LLM:', str(res)[:300])
    use_llm = not any(w in str(res).lower() for w in ('error', 'fail', 'not found', 'false'))
except Exception as e:
    print('LLM off:', e)
for j in jobs:
    t = time.time()
    try:
        p = GenerationParams(caption=j['caption'], lyrics=j['lyrics'], vocal_language='hi', duration=j['duration'], bpm=j['bpm'], thinking=use_llm, inference_steps=8)
        c = GenerationConfig(batch_size=2, audio_format='mp3')
        r = generate_music(dit, llm, p, c, save_dir='/content/out/' + j['name'])
        print(j['name'], 'OK' if r.success else r.error, round(time.time() - t), 's', flush=True)
    except Exception:
        traceback.print_exc()
"""
open('/content/gen.py', 'w').write(script)
jobs = [
 {"name": "chhath", "bpm": 80, "duration": 90,
  "caption": "Traditional Bhojpuri Chhath folk devotional song, warm rich female village singer, soft dholak, harmonium, shehnai, manjira, gentle morning river ghat atmosphere, slow 80 BPM, emotional and pure",
  "lyrics": "[verse]\nउगा हो सूरज देव, भइल अरघ के बेर\nछठी मइया के घाटे, जगमग दियरा के फेर\n[chorus]\nजय छठी मइया, जय हो सूरज देव\nजय छठी मइया, जय हो सूरज देव\n[verse]\nसूपवा में फल फूल, अँचरा में सनेह\nपरिवार के सुख माँगीं, माई देहु असीस\n[chorus]\nजय छठी मइया, जय हो सूरज देव"},
 {"name": "hanuman", "bpm": 90, "duration": 90,
  "caption": "Hindi devotional bhajan, deep warm male singer, harmonium, tabla, manjira, temple bells, chorus response, steady 90 BPM, peaceful and powerful",
  "lyrics": "[verse]\nश्रीगुरु चरन सरोज रज, निज मनु मुकुरु सुधारि\nबरनउँ रघुबर बिमल जसु, जो दायकु फल चारि\n[chorus]\nजय हनुमान ज्ञान गुन सागर\nजय कपीस तिहुँ लोक उजागर\n[verse]\nराम दूत अतुलित बल धामा\nअंजनि पुत्र पवनसुत नामा\n[chorus]\nजय हनुमान ज्ञान गुन सागर\nजय कपीस तिहुँ लोक उजागर"}
]
import json
json.dump(jobs, open('/content/jobs.json', 'w'), ensure_ascii=False)
print('✅ ready')

In [ ]:
#@title 4 · गाने बनाएँ (3–6 मिनट)
!python /content/gen.py 2>&1 | grep -E "OK|Error|error|LLM|Traceback|s$" | tail -20

In [ ]:
#@title 5 · सुनें 🎧
import glob
from IPython.display import Audio, display, HTML
files = sorted(glob.glob('/content/out/**/*.mp3', recursive=True))
if not files: print('❌ कोई गाना नहीं बना — ऊपर वाले cell का error मुझे भेजें')
for f in files:
    display(HTML(f'<b>{f.split("/")[-2]}</b> — {f.split("/")[-1]}'))
    display(Audio(f))

सुनकर Claude को बताइए: **आवाज़ कैसी है, Hindi शब्द साफ़ हैं या नहीं, Suno के मुक़ाबले कैसा है।**